# 06 — Machine Learning : Ridge & Lasso
## Comparaison avec la régression OLS classique

**Objectif** : améliorer la robustesse du modèle OLS en appliquant des régressions  
régularisées (Ridge et Lasso) et comparer leurs performances.

---

### Rappel : OLS, Ridge et Lasso — quelle différence ?

| Modèle | Principe | Quand l'utiliser |
|--------|----------|------------------|
| **OLS** | Minimise la somme des erreurs au carré | Pas de multicolinéarité, peu de variables |
| **Ridge** | OLS + pénalité sur la taille des coefficients (L2) | Beaucoup de variables corrélées — rétrécit les coefficients sans les annuler |
| **Lasso** | OLS + pénalité qui force certains coefficients à zéro (L1) | Sélection automatique de variables — élimine les moins importantes |

Le paramètre **alpha** contrôle la force de la régularisation :  
- `alpha = 0` → équivalent à l'OLS  
- `alpha élevé` → coefficients très réduits (Ridge) ou beaucoup de zéros (Lasso)

---

## 0. Imports et configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import RidgeCV, LassoCV, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, root_mean_squared_error
from pathlib import Path

plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.size'] = 11

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUTPUTS_DIR = PROJECT_DIR / 'outputs'
DATA_DIR    = Path('C:/Users/Enes/data')
OUTPUTS_DIR.mkdir(exist_ok=True)

print('✓ Imports OK')
print('Outputs :', OUTPUTS_DIR)

## 1. Chargement et préparation des données

On reconstruit le même dataset que le notebook 04 :  
agrégation par **année × classe ATC1**, avec dummies ATC et standardisation des variables.

In [ ]:
def parse_euro(s):
    return (s.astype(str).str.strip()
             .str.replace('.', '', regex=False)
             .str.replace(',', '.', regex=False)
             .replace('', '0').astype(float))

def load_year(annee):
    for pat in [f'OPEN_MEDIC_{annee}.zip', f'OPEN_MEDIC_{annee}.csv']:
        p = DATA_DIR / pat
        if p.exists():
            break
    else:
        return None
    df = pd.read_csv(p, sep=None, engine='python', encoding='latin-1')
    df.columns = df.columns.str.lower().str.strip()
    for col in ['l_atc1', 'l_cip13']:
        if col not in df.columns: df[col] = ''
    for col in ['rem', 'bse']:
        df[col] = parse_euro(df[col])
    df['annee'] = annee
    return df

frames = [load_year(a) for a in range(2016, 2026)]
df_raw = pd.concat([f for f in frames if f is not None], ignore_index=True)
print(f'✓ {df_raw.shape[0]:,} lignes chargées')

In [ ]:
# Agrégation année × ATC1
df_agg = (
    df_raw[df_raw['l_atc1'] != '']
    .groupby(['annee', 'l_atc1'])
    .agg(rem_total=('rem', 'sum'), boites_total=('boites', 'sum'))
    .reset_index()
)

# Dummies ATC1
dummies = pd.get_dummies(df_agg['l_atc1'], prefix='ATC', drop_first=True)
df_model = pd.concat([df_agg[['annee', 'boites_total', 'rem_total']], dummies], axis=1)
df_model = df_model.dropna()[df_model['rem_total'] > 0]

X_cols = ['annee', 'boites_total'] + [c for c in df_model.columns if c.startswith('ATC_')]
X_raw  = df_model[X_cols].astype(float).values
y      = df_model['rem_total'].astype(float).values

# Standardisation (obligatoire pour Ridge/Lasso — met toutes les X à la même échelle)
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print(f'✓ Dataset prêt : {X_scaled.shape[0]} observations × {X_scaled.shape[1]} variables')
print(f'  Classes ATC1 : {df_agg["l_atc1"].nunique()}')
print(f'  Années       : {sorted(df_agg["annee"].unique())}')

## 2. OLS — modèle de référence

On commence par réestimer l'OLS avec scikit-learn pour avoir une **baseline comparable**  
aux modèles Ridge et Lasso (même échelle, même split train/test).

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

ols = LinearRegression()
ols.fit(X_scaled, y)
y_pred_ols = ols.predict(X_scaled)

r2_ols   = r2_score(y, y_pred_ols)
rmse_ols = root_mean_squared_error(y, y_pred_ols)

print('=== OLS (référence) ===')
print(f'R²   : {r2_ols:.4f}')
print(f'RMSE : {rmse_ols:,.0f} €')

---
## 3. Ridge — régularisation L2

**Comment fonctionne Ridge ?**  
Ridge ajoute une pénalité proportionnelle au **carré** de chaque coefficient :  

$$\min \sum(y_i - \hat{y}_i)^2 + \alpha \sum \beta_j^2$$

Résultat : tous les coefficients sont **réduits** vers zéro, mais aucun n'est exactement nul.  
Ridge est idéal quand toutes les variables contribuent un peu, même faiblement.

**RidgeCV** teste automatiquement plusieurs valeurs d'alpha et garde le meilleur  
par validation croisée.

In [ ]:
alphas_ridge = np.logspace(-3, 6, 100)  # de 0.001 à 1,000,000

ridge = RidgeCV(alphas=alphas_ridge, cv=5, scoring='r2')
ridge.fit(X_scaled, y)
y_pred_ridge = ridge.predict(X_scaled)

r2_ridge   = r2_score(y, y_pred_ridge)
rmse_ridge = root_mean_squared_error(y, y_pred_ridge)

print('=== RIDGE ===')
print(f'Meilleur alpha : {ridge.alpha_:.4f}')
print(f'R²             : {r2_ridge:.4f}')
print(f'RMSE           : {rmse_ridge:,.0f} €')
print(f'Gain R² vs OLS : {(r2_ridge - r2_ols):+.4f}')

In [ ]:
# Courbe de sélection d'alpha Ridge
r2_par_alpha = []
for a in alphas_ridge:
    from sklearn.linear_model import Ridge
    m = Ridge(alpha=a).fit(X_scaled, y)
    r2_par_alpha.append(r2_score(y, m.predict(X_scaled)))

plt.figure()
plt.semilogx(alphas_ridge, r2_par_alpha, color='steelblue', linewidth=2)
plt.axvline(ridge.alpha_, color='red', linestyle='--', label=f'Meilleur alpha = {ridge.alpha_:.2f}')
plt.title('Ridge — R² selon la valeur d\'alpha')
plt.xlabel('Alpha (échelle log)')
plt.ylabel('R²')
plt.legend()
plt.tight_layout()
plt.show()

---
## 4. Lasso — régularisation L1 et sélection de variables

**Comment fonctionne Lasso ?**  
Lasso ajoute une pénalité proportionnelle à la **valeur absolue** de chaque coefficient :

$$\min \sum(y_i - \hat{y}_i)^2 + \alpha \sum |\beta_j|$$

Résultat : certains coefficients sont forcés **exactement à zéro** — Lasso fait  
de la **sélection automatique de variables**.  
C'est très utile ici car nos dummies ATC peuvent être redondantes ou peu informatives.

**LassoCV** trouve automatiquement le meilleur alpha par validation croisée.

In [ ]:
lasso = LassoCV(cv=5, random_state=42, max_iter=10000, n_alphas=100)
lasso.fit(X_scaled, y)
y_pred_lasso = lasso.predict(X_scaled)

r2_lasso   = r2_score(y, y_pred_lasso)
rmse_lasso = root_mean_squared_error(y, y_pred_lasso)

coefs_lasso = pd.Series(lasso.coef_, index=X_cols)
n_zeros     = (coefs_lasso == 0).sum()
n_actifs    = (coefs_lasso != 0).sum()

print('=== LASSO ===')
print(f'Meilleur alpha     : {lasso.alpha_:.4f}')
print(f'R²                 : {r2_lasso:.4f}')
print(f'RMSE               : {rmse_lasso:,.0f} €')
print(f'Variables actives  : {n_actifs} / {len(coefs_lasso)}')
print(f'Variables éliminées: {n_zeros} (coef = 0)')

In [ ]:
print('=== VARIABLES ÉLIMINÉES PAR LASSO (coef = 0) ===')
eliminees = coefs_lasso[coefs_lasso == 0].index.tolist()
if eliminees:
    for v in eliminees:
        print(f'  ✗ {v}')
else:
    print('  Aucune variable éliminée (alpha très faible)')

print()
print('=== VARIABLES LES PLUS IMPORTANTES (coef ≠ 0, triées) ===')
actives = coefs_lasso[coefs_lasso != 0].abs().sort_values(ascending=False)
print(actives.head(10).to_string())

In [ ]:
# Chemin de régularisation Lasso : évolution des coefficients selon alpha
from sklearn.linear_model import lasso_path

alphas_path, coefs_path, _ = lasso_path(X_scaled, y, n_alphas=80)

plt.figure(figsize=(13, 6))
for i, name in enumerate(X_cols):
    plt.semilogx(alphas_path, coefs_path[i], linewidth=1.2, alpha=0.8)
plt.axvline(lasso.alpha_, color='red', linestyle='--', linewidth=1.5,
            label=f'Alpha optimal = {lasso.alpha_:.4f}')
plt.title('Lasso — Chemin de régularisation (évolution des coefficients)')
plt.xlabel('Alpha (échelle log)')
plt.ylabel('Coefficient')
plt.axhline(0, color='black', linewidth=0.5)
plt.legend()
plt.tight_layout()
plt.show()

---
## 5. Comparaison des coefficients : OLS vs Ridge vs Lasso

Ce graphique montre comment chaque méthode **traite différemment** les mêmes variables :  
- **OLS** : coefficients libres, potentiellement instables si multicolinéarité  
- **Ridge** : coefficients réduits proportionnellement à leur taille  
- **Lasso** : certains coefficients exactement à zéro (sélection)

In [ ]:
df_coefs = pd.DataFrame({
    'OLS'  : ols.coef_,
    'Ridge': ridge.coef_,
    'Lasso': lasso.coef_,
}, index=X_cols)

# Sélectionne les variables les plus discriminantes (variance des coefs élevée)
df_coefs['variance'] = df_coefs.var(axis=1)
top_vars = df_coefs['variance'].nlargest(12).index
df_plot  = df_coefs.loc[top_vars, ['OLS', 'Ridge', 'Lasso']]

x     = np.arange(len(df_plot))
width = 0.28

fig, ax = plt.subplots(figsize=(14, 7))
ax.bar(x - width, df_plot['OLS'],   width, label='OLS',   color='steelblue', alpha=0.85)
ax.bar(x,         df_plot['Ridge'], width, label='Ridge', color='seagreen',  alpha=0.85)
ax.bar(x + width, df_plot['Lasso'], width, label='Lasso', color='tomato',    alpha=0.85)
ax.axhline(0, color='black', linewidth=0.7)
ax.set_xticks(x)
ax.set_xticklabels([v.replace('ATC_','') for v in df_plot.index],
                   rotation=35, ha='right', fontsize=9)
ax.set_title('Comparaison des coefficients : OLS vs Ridge vs Lasso\n(12 variables les plus discriminantes)')
ax.set_ylabel('Coefficient (données standardisées)')
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. Tableau récapitulatif des performances

Le **R²** mesure la part de variance expliquée (plus proche de 1 = mieux).  
Le **RMSE** mesure l'erreur moyenne en euros (plus bas = mieux).

In [ ]:
# Récupère le R² OLS statsmodels depuis outputs/ si dispo
ols_sm_path = OUTPUTS_DIR / 'metriques_ols.csv'
r2_ols_sm   = pd.read_csv(ols_sm_path)['R2'].iloc[0] if ols_sm_path.exists() else r2_ols

tableau = pd.DataFrame([
    {
        'Modèle'              : 'OLS (statsmodels)',
        'Alpha'               : '-',
        'R²'                  : round(r2_ols_sm, 4),
        'RMSE (€)'            : '-',
        'Variables actives'   : len(X_cols),
        'Variables éliminées' : 0,
        'Régularisation'      : 'Aucune',
    },
    {
        'Modèle'              : 'OLS (sklearn)',
        'Alpha'               : '-',
        'R²'                  : round(r2_ols, 4),
        'RMSE (€)'            : f'{rmse_ols:,.0f}',
        'Variables actives'   : len(X_cols),
        'Variables éliminées' : 0,
        'Régularisation'      : 'Aucune',
    },
    {
        'Modèle'              : 'Ridge (RidgeCV)',
        'Alpha'               : round(ridge.alpha_, 4),
        'R²'                  : round(r2_ridge, 4),
        'RMSE (€)'            : f'{rmse_ridge:,.0f}',
        'Variables actives'   : len(X_cols),
        'Variables éliminées' : 0,
        'Régularisation'      : 'L2 (rétrécissement)',
    },
    {
        'Modèle'              : 'Lasso (LassoCV)',
        'Alpha'               : round(lasso.alpha_, 4),
        'R²'                  : round(r2_lasso, 4),
        'RMSE (€)'            : f'{rmse_lasso:,.0f}',
        'Variables actives'   : int(n_actifs),
        'Variables éliminées' : int(n_zeros),
        'Régularisation'      : 'L1 (sélection)',
    },
])

print(tableau.to_string(index=False))

# Export
tableau.to_csv(OUTPUTS_DIR / 'comparaison_modeles.csv', index=False, encoding='utf-8')
df_coefs.round(4).to_csv(OUTPUTS_DIR / 'comparaison_coefficients.csv', encoding='utf-8')
print('\n✓ Exporté dans outputs/ :')
print('  - comparaison_modeles.csv')
print('  - comparaison_coefficients.csv')

---
## 7. Conclusion

### Que retenir ?

**Ridge** convient mieux quand les variables sont corrélées entre elles (ce qui est le cas  
avec des dummies ATC issues d'une même classification). Il stabilise les coefficients  
sans en éliminer aucun.

**Lasso** est utile pour identifier les classes ATC **vraiment discriminantes** :  
les variables éliminées (coef = 0) n'apportent pas d'information significative  
une fois les autres contrôlées.

Si les trois modèles donnent des R² proches, cela indique que le modèle OLS  
était déjà bien spécifié et que la régularisation apporte surtout de la robustesse,  
pas un gain de prédiction.

### Prochaine étape
Pour améliorer significativement les prédictions, envisager :
- **Random Forest / Gradient Boosting** : captent les non-linéarités
- **Données de panel avec effets fixes** : éliminent l'endogénéité intra-classe
- **Ajout de variables exogènes** : démographie, prix ANSM, indicateurs épidémiologiques